# QuickQuake catalog analysis: Pavlof 2021 example

Run `QQ_driver.py` in this folder first: it downloads 16 hours of data (2021-07-29 08:00 to 2021-07-30 00:00 UTC) and writes the results in `data/` next to this notebook.

Analyses of a QuickQuake catalog used in the papers. All the logic lives in `qq_helpers.py`;
this notebook only sets paths, loads the results, calls the analysis functions and (optionally)
saves the figures in `DATA_ROOT/images/analysis/`. Edit the **Settings** cell, then run all
cells. A section whose inputs you do not have (an FI catalog, reference catalogs such as USGS
or AVO, enough shallow events for DBSCAN) is skipped with a message. The general figures are in
`catalog_visualization.ipynb` (`visualization.ipynb` in quickquake/).

**Kernel:** the QuickQuake conda environment (the one `install.sh` creates). In VS Code select it as the kernel; in a terminal run `conda activate <environment>` and then `jupyter lab`.

| Section | Analysis |
|---|---|
| 2 | Magnitude of completeness (cumulative FMD, magnitude vs time) |
| 3 | Frequency index (FI) composition by month |
| 4 | Comparison with reference catalogs (matching, residuals, overview, maps) |
| 5 | DBSCAN clusters, best-fitting surfaces, dip and dip direction |

In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio

# This notebook lives in <QuickQuake>/examples/pavlof_2021/; qq_helpers.py is in <QuickQuake>/quickquake/
HELPERS_DIR = (Path.cwd() / "../../quickquake").resolve()
if str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

import qq_helpers as h
importlib.reload(h)
print("helpers:", h.__file__)

pio.renderers.default = "plotly_mimetype+notebook_connected"   # interactive figures in VS Code and Jupyter

## 0. Settings
Everything you normally edit is in this cell.

In [ ]:
# ---------------- results of a QuickQuake run ----------------
# Relative paths are relative to this notebook's folder (examples/pavlof_2021/).
DATA_ROOT = Path("data").resolve()                           # written by QQ_driver.py in this folder
NAMEBASE = "GAMMA"                                           # same as location_namebase in QQ_driver.py
REGION = {"name": "Pavlof", "lat": 55.4133, "lon": -161.8903}   # volcano / region centre (degrees)
RADIUS_KM = None              # keep only events within this distance (km) of the centre; None = keep all
START = "2021-07-29T08:00:00Z"     # same window as START / END in QQ_driver.py
END   = "2021-07-30T00:00:00Z"

# ---------------- what to draw ----------------
SAVE_FIGURES = True           # save into DATA_ROOT/images/analysis/
DEPTH_MAX_KM = 40.0           # deepest event shown in depth-coloured / 3-D figures (km)
SPANS = []                    # (start, end, label) periods, e.g. ("2021-08-05", "2021-08-06", "Ash emissions")

# ---------------- optional inputs (None = skip that section) ----------------
FI_CATALOG_CSV = None         # catalog with an "FI" column (VT / LP / NS) from frequency_index/
REFERENCE_CATALOGS = {}       # e.g. {"usgs": "comcatalog.csv", "avo": "earthquakesavo.csv"} for section 4
COMPARISON_YEAR = 2021        # section 4 compares one calendar year
MATCH_TIME_TOL_S = 1.5        # section 4: |origin-time difference| allowed (s)
MATCH_DIST_TOL_KM = 8.0       # section 4: 3-D distance allowed (km)

# ---------------- section 5: DBSCAN clusters and surfaces ----------------
DBSCAN_DEPTH_MAX_KM = 10.0                       # only events shallower than this go into DBSCAN (km)
DBSCAN_EPS_VALUES = np.arange(0.182, 0.195, 0.002)   # standard-deviation units of the standardised coordinates
DBSCAN_MIN_SAMPLES_VALUES = [15, 20, 25]
DBSCAN_MIN_CLUSTER_SIZE = 100                    # a "large" cluster has at least this many events
CLUSTER_SELECTION = dict(mode="top_n", top_n=3, manual_ids=[0, 1], min_events=100)
VIEW_3D = dict(exagg=2.0, view_elev=5.0, view_azim=265.0)

# ---------------- paths inside DATA_ROOT (standard QuickQuake layout) ----------------
MERGED = DATA_ROOT / "merged"
CATALOG_CSV   = MERGED / f"output/{NAMEBASE}_hypodd_catalog_qcfiltered.csv"   # final QuickQuake catalog
HYPODD_CSV    = MERGED / f"output/{NAMEBASE}_hypodd_catalog.csv"
HYPOINV_CSV   = MERGED / f"output/{NAMEBASE}_hypoinv_good.csv"
GAMMA_CSV     = MERGED / f"input/{NAMEBASE}_catalog_cleaned.csv"
PICKS_CSV     = MERGED / f"input/{NAMEBASE}_picks_cleaned.csv"
STATIONS_JSON = MERGED / f"input/{NAMEBASE}_station_list.json"
WAVEFORM_INDEX = DATA_ROOT / "waveform_index.pkl"


IMAGE_DIR = h.ensure_dir(DATA_ROOT / "images" / "analysis")


def save(fig, name, **kwargs):
    """Save a figure into IMAGE_DIR when SAVE_FIGURES is True."""
    if SAVE_FIGURES and fig is not None:
        out = IMAGE_DIR / name
        if hasattr(fig, "write_html"):
            fig.write_html(out, include_plotlyjs="cdn")
        else:
            fig.savefig(out, bbox_inches="tight", **kwargs)
        print("saved:", out)

## 1. Load the catalog and the stations

In [ ]:
catc = h.CatalogCols(time="time", lat="latitude", lon="longitude", depth_km="depth_km", mag="magnitude")
stc = h.StationCols(station_id="station", lat="latitude", lon="longitude", elev_km="elev_km")

df_catalog = h.load_catalog_csv(CATALOG_CSV, catc=catc)
df_stations = h.load_stations_json(STATIONS_JSON)
print(f"catalog: {len(df_catalog)} events | stations: {len(df_stations)}")

if RADIUS_KM is not None:
    df_catalog = h.filter_catalog_by_radius(df_catalog, center_lat=REGION["lat"], center_lon=REGION["lon"],
                                            radius_km=RADIUS_KM, lat_col=catc.lat, lon_col=catc.lon)
    print(f"within {RADIUS_KM:g} km of {REGION['name']}: {len(df_catalog)} events "
          f"(farthest {df_catalog['r_km'].max():.2f} km)")

print("time range:", df_catalog[catc.time].min(), "->", df_catalog[catc.time].max())

## 2. Magnitude of completeness
Mc by maximum curvature (MAXC) on bins of 0.1; the red line is a least-squares Gutenberg-Richter fit above Mc (slope ~ -b).

In [ ]:
fig, ax, mc_results = h.plot_fmd_cumulative(df_catalog, mag_col=catc.mag)
r = mc_results["Catalog"]
print(f"{r['n_events']} events | Mc (MAXC) = {r['mc']:.2f} | G-R slope = {r['slope']:.3f}")
save(fig, "mc_cumulative.png", dpi=300)
plt.show()

fig, ax, mcs = h.plot_magnitude_time(df_catalog, mag_col=catc.mag, time_col=catc.time)
save(fig, "mc_magnitude_time.png", dpi=300)
plt.show()

## 3. Frequency-index (FI) composition by month

In [ ]:
if FI_CATALOG_CSV is None or not Path(FI_CATALOG_CSV).exists():
    print("FI section skipped: set FI_CATALOG_CSV to a catalog with an FI column")
else:
    df_fi = pd.read_csv(FI_CATALOG_CSV)
    # the paper 2 look ("talk" + "whitegrid") from matplotlib's own style sheets, only in this section
    with plt.style.context(["seaborn-v0_8-whitegrid", "seaborn-v0_8-talk"]):
        fig, axes, fi_info = h.plot_fi_monthly_composition(df_fi, spans=SPANS, title=REGION["name"])
        save(fig, "fi_counts_percentage_spans.svg")
        plt.show()
        fig_leg, _ = h.plot_monthly_fi_span_legend_only(
            fi_colors=h.FI_COMPOSITION_COLORS, fi_order=list(fi_info["counts"].columns),
            span_color_map=fi_info["span_color_map"], span_order=list(dict.fromkeys(s[2] for s in SPANS)),
            figsize=(4.5, 3.2), legend_fontsize=16)
        save(fig_leg, "fi_span_legend.svg")
        plt.show()
    print(f"{len(df_fi)} events | months on the axis: {len(fi_info['counts'])}")

## 4. Comparison with reference catalogs
Events are matched when |Δt| ≤ `MATCH_TIME_TOL_S` and the 3-D distance ≤ `MATCH_DIST_TOL_KM`
(one-to-one, greedy by score). Residuals are QuickQuake − reference (Δz > 0: QuickQuake deeper).
All catalogs are cut to `COMPARISON_YEAR` and to `RADIUS_KM` around the region centre.

In [ ]:
PIPELINE_MAPPING = {"time": "time", "latitude": "latitude", "longitude": "longitude", "depth_km": "depth_km",
                    "magnitude": "mag"}
CATALOGS = {
    "gamma":   {"file": GAMMA_CSV, "label": "GAMMA", "depth_in_meters": True,
                "mapping": {**PIPELINE_MAPPING, "depth(m)": "depth_km", "event_id_mapped": "event_id"}},
    "hypoinv": {"file": HYPOINV_CSV, "label": "Hypoinverse", "mapping": {**PIPELINE_MAPPING, "event_id_mapped": "event_id"}},
    "hypodd":  {"file": HYPODD_CSV, "label": "HypoDD", "mapping": {**PIPELINE_MAPPING, "event_id": "event_id"}},
    "qq":      {"file": CATALOG_CSV, "label": "QuickQuake", "color": "#4C78A8", "scatter_size": 2.5, "scatter_alpha": 0.3,
                "mapping": {**PIPELINE_MAPPING, "event_id": "event_id"}},
    "usgs":    {"file": REFERENCE_CATALOGS.get("usgs"), "label": "USGS", "color": "red", "scatter_size": 3.5, "scatter_alpha": 0.7,
                "mapping": {"time": "time", "latitude": "latitude", "longitude": "longitude", "depth": "depth_km",
                            "mag": "mag", "id": "event_id"}},          # USGS ComCat CSV
    "avo":     {"file": REFERENCE_CATALOGS.get("avo"), "label": "AVO", "color": "#54A24B", "scatter_size": 3.5, "scatter_alpha": 0.7,
                "mapping": {"Event ID": "event_id", "Date/Time UTC": "time", "Depth km": "depth_km", "Magnitude": "mag",
                            "Latitude": "latitude", "Longitude": "longitude"}},   # AVO catalog search CSV
}
CATALOGS = {k: c for k, c in CATALOGS.items() if c["file"] is not None and Path(c["file"]).exists()}
references = [k for k in ("usgs", "avo") if k in CATALOGS]

if not references:
    print("comparison section skipped: set REFERENCE_CATALOGS in the Settings cell")
else:
    catalog_buf = {}
    for key, cfg in CATALOGS.items():
        df = h.load_and_normalize(cfg["file"], cfg["mapping"], depth_in_meters=cfg.get("depth_in_meters", False))
        df = h.filter_catalog_by_year(df, COMPARISON_YEAR)
        if RADIUS_KM is not None:
            df = h.filter_catalog_by_radius(df, center_lat=REGION["lat"], center_lon=REGION["lon"], radius_km=RADIUS_KM)
        catalog_buf[key] = df
        print(f"{cfg['label']:12s} {len(df):6d} events in {COMPARISON_YEAR}")

    comparisons = [{"ref": ref, "qry": qry, "name": f"{CATALOGS[qry]['label']} vs {CATALOGS[ref]['label']}"}
                   for qry in ("gamma", "hypoinv", "hypodd", "qq") if qry in CATALOGS for ref in references]
    if {"usgs", "avo"} <= set(references):
        comparisons.append({"ref": "avo", "qry": "usgs", "name": "USGS vs AVO"})
    pair_results = h.run_all_comparisons(catalog_buf, comparisons, CATALOGS,
                                         {"time_tol_s": MATCH_TIME_TOL_S, "dist_tol_km": MATCH_DIST_TOL_KM}, verbose=False)
    metrics_table = h.build_metrics_table(pair_results)
    display(metrics_table)

In [ ]:
if references:
    for ref in references:
        matched = pair_results[(ref, "qq")]["matches"]
        name = f"QuickQuake vs {CATALOGS[ref]['label']}"
        display(h.build_km_component_metrics_table(matched))
        fig, axes = h.plot_match_histograms(matched, comparison_name=name, show=False)
        save(fig, f"match_histograms_qq_vs_{ref}.png", dpi=300); plt.show()
        fig, axes = h.plot_match_km_histograms(matched, comparison_name=f"{name} (Δx, Δy, Δz, Δt)", show=False)
        save(fig, f"residuals_km_qq_vs_{ref}.png", dpi=300); plt.show()
        fig, axes = h.plot_match_km_scatter(matched, comparison_name=f"{name} signed residuals in km", show=False)
        save(fig, f"residuals_scatter_qq_vs_{ref}.png", dpi=300); plt.show()

In [ ]:
if {"usgs", "avo"} <= set(references):
    fig, axes, residual_data = h.plot_combined_reference_residual_histograms(
        pair_results, bins=32, use_density=False, spatial_plot_limit_km=10.0, time_plot_limit_s=2.0, show=False)
    save(fig, "combined_residual_histograms_qq_usgs_avo.png", dpi=600)
    plt.show()
    component_table, paragraph_table = h.make_residual_summary_tables(residual_data)
    display(component_table)
    display(paragraph_table)

In [ ]:
if references:
    fig, axes = h.plot_three_catalog_overview_pretty(
        catalog_plot_dict={k: catalog_buf[k] for k in ("qq", *references)}, configs=CATALOGS, volcano=REGION,
        year=COMPARISON_YEAR, radius_km=RADIUS_KM, time_bin="5D", max_depth_km=DEPTH_MAX_KM, show=False)
    save(fig, "catalog_overview.svg")
    plt.show()

In [ ]:
if references:
    # QuickQuake and the first reference catalog on maps with the same limits
    ref = references[0]
    year_start, year_end = f"{COMPARISON_YEAR}-01-01T00:00:00Z", f"{COMPARISON_YEAR}-12-31T23:59:59Z"
    xlim_common, ylim_common = h.get_common_bounds_for_two_catalogs(
        catalog_buf["qq"], catalog_buf[ref], year_start, year_end, pad_lon=0.08, pad_lat=0.06, zoom_factor=1.8)
    for key in ("qq", ref):
        fig, ax, meta = h.plot_mapview_seismicity_zoom(
            catalog=catalog_buf[key], start=year_start, end=year_end, depth_vmin=1.0, depth_vmax=DEPTH_MAX_KM,
            cmap_name="gist_rainbow", size_k=20.0, relief_zoom=12, step_lon=0.05, step_lat=0.05, labelsize=18,
            figsize=(10, 8), xlim=xlim_common, ylim=ylim_common, show=False)
        save(fig, f"mapview_common_limits_{key}.svg")
        plt.show()
        print(f"{CATALOGS[key]['label']}: {meta['n_events']} events drawn of {len(catalog_buf[key])} "
              "(add depth_filter_min / depth_filter_max to hide a depth range; reference catalogs "
              "have negative depths = above sea level)")

In [ ]:
if references:
    # Mc of QuickQuake and the reference catalogs over their common time window
    common_end = min(catalog_buf[k]["time"].max() for k in ("qq", *references))
    mc_catalogs = {CATALOGS[k]["label"]: catalog_buf[k][catalog_buf[k]["time"] <= common_end] for k in ("qq", *references)}
    print("common window ends:", common_end, "| magnitudes may be on different scales (e.g. ML vs QuickQuake)")
    fig, ax, mc_cmp = h.plot_fmd_cumulative(mc_catalogs, mag_col="mag")
    save(fig, "mc_cumulative_vs_reference.png", dpi=300); plt.show()
    fig, ax, _ = h.plot_magnitude_time(mc_catalogs, mag_col="mag", title=f"{REGION['name']} {COMPARISON_YEAR}: magnitudes through time")
    save(fig, "mc_magnitude_time_vs_reference.png", dpi=300); plt.show()

## 5. DBSCAN clusters and best-fitting surfaces
DBSCAN runs on local-km coordinates standardised per axis, so `eps` is in standard-deviation
units, not km. For each selected cluster the simplest polynomial surface (plane, quadratic or
cubic) within 5 % of the best cross-validated RMSE is kept; dip and dip direction are those of
the cluster's best-fitting plane.

In [ ]:
df_db = df_catalog.copy()
df_db = df_db[(df_db[catc.time] >= pd.to_datetime(START, utc=True)) & (df_db[catc.time] <= pd.to_datetime(END, utc=True))]
df_db = df_db[df_db[catc.depth_km] <= DBSCAN_DEPTH_MAX_KM].dropna(subset=[catc.lon, catc.lat, catc.depth_km]).reset_index(drop=True)
print("events going into DBSCAN:", len(df_db))

scan = h.dbscan_parameter_scan(df_db[[catc.lon, catc.lat, catc.depth_km]].to_numpy(float), DBSCAN_EPS_VALUES,
                               DBSCAN_MIN_SAMPLES_VALUES, min_cluster_size=DBSCAN_MIN_CLUSTER_SIZE)
try:
    best = h.select_best_dbscan(scan["results"])
except ValueError as err:
    best = None
    print("cluster section skipped:", err)

In [ ]:
if best is not None:
    df_db["cluster_dbscan"] = best["labels"]
    cluster_table = h.dbscan_cluster_table(df_db, best["labels"], catc=catc)
    display(cluster_table)
    cluster_ids = h.select_cluster_ids(cluster_table, **CLUSTER_SELECTION)
    print("selected clusters:", cluster_ids)

    surface_fits, surface_meshes, cluster_points, surface_table = h.fit_cluster_surfaces(
        df_db, cluster_ids, scan["lon_ref"], scan["lat_ref"], catc=catc)
    display(surface_table)
    if SAVE_FIGURES:
        surface_table.to_csv(IMAGE_DIR / "cluster_surfaces_dip_dipdir.csv", index=False)

    styles = h.cluster_style_map(cluster_ids)
    fig, ax, meta = h.plot_cluster_surfaces_3d(df_db, cluster_ids, cluster_points, surface_meshes, START, END,
                                               catc=catc, cluster_styles=styles, depth_max=DBSCAN_DEPTH_MAX_KM, **VIEW_3D)
    save(fig, "cluster_surfaces_3d.png", dpi=300)
    plt.show()

    fig = h.plot_cluster_surfaces_interactive(cluster_ids, cluster_points, surface_meshes, surface_fits, surface_table,
                                              scan["lon_ref"], scan["lat_ref"], cluster_styles=styles)
    save(fig, "cluster_surfaces_interactive.html")
    fig.show()